In [1]:


#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd 

from time import sleep

import datetime

from pandas import ExcelWriter

import string

import re

import pdfplumber

import os

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.alert import Alert

In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'ID BI' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running ID BI Web Scraping Tool v.1.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()




# %%

In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        regulatorName + ' 1': 'https://www.bi.go.id/en/layanan/informasi-perizinan/pasar-keuangan/default.aspx',
        regulatorName + ' 2': 'https://www.bi.go.id/en/layanan/informasi-perizinan/sistem-pembayaran/default.aspx',
        }



Typology={

       regulatorName + ' 1': 'Money Market',
       regulatorName + ' 2': 'Payment System',


        }




sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 'RegCtry': [], 'RegCode' : [], 'ListCode': [], 
         'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 'Phone - Mother company': [], 'Check': []}




now = datetime.datetime.now()

processdate = now.strftime('%Y-%m-%d')




In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


In [6]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for reg in regdict:
    
    
    print(f'Working with list {reg}')
    driver.get(regdict[reg])
    sleep(2)
    soup = BeautifulSoup(driver.page_source, 'html.parser')  
    sleep(5)
    if reg == 'ID BI 1':
        driver.find_element(By.CLASS_NAME, 'btn-export').click()
    elif reg == 'ID BI 2':
        button = driver.find_element(By.XPATH, '//*[@id="ButtonExport"]')
        driver.execute_script("arguments[0].click();", button)
    
            
    sleep(2)
    dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
    try:
        sleep(5)
        if dl_files[0].endswith('.xlsx'):
            dataframe = pd.read_excel(dl_files[0], engine='openpyxl')
        else :
            status = dl_files[0].endswith('.xlsx')
            while not status:
                sleep(5)
                status = dl_files[0].endswith('.xlsx')
                dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            dataframe = pd.read_excel(dl_files[0], engine='openpyxl')
    except:
        sleep(5)
        if dl_files[0].endswith('.xlsx'):
            dataframe = pd.read_excel(dl_files[0], engine='openpyxl')
        else :
            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
            dataframe = pd.read_excel(dl_files[0], engine='openpyxl')
    dataframe = dataframe.fillna('')
    if reg == 'ID BI 1':
        dataframe.columns = dataframe.iloc[3]
        dataframe = dataframe[4:]  # Remove the first row which is now the header
        
        sqldict = bourange_same_length_array(sqldict)   
        df1=pd.DataFrame(sqldict)
        
        licensed_dataframe = dataframe[dataframe['Status'] == '1'].copy()
        if not licensed_dataframe.empty:
            df1['InternalID_1'] = licensed_dataframe['Letter Number']
            df1['InternalID_1_type'] = 'Letter Number'
            licensed_dataframe.loc[:, 'Description EN'] = licensed_dataframe['Description EN'].apply(lambda x: x.replace('License Number: ', '') if 'License Number' in x else x)
            df1['InternalID_2'] = licensed_dataframe['Description EN']
            df1['InternalID_2_type'] = 'Description ID'
            df1['Name'] = licensed_dataframe['Name']
            df1['ListProcessDate'] = processdate
            df1['RegCtry'] = reg.split(' ')[0]
            df1['RegCode'] = reg.split(' ')[1]
            df1['ListCode'] = reg.split(' ')[-1]
            df1['ListName'] = Typology[reg]
            df1['RegulationType'] = 'Regulated' 
        df1.fillna('')
            
    elif reg == 'ID BI 2':
        dataframe.columns = dataframe.iloc[3]
        dataframe = dataframe[4:]  # Remove the first row which is now the header
        
        sqldict = bourange_same_length_array(sqldict)   
        df2=pd.DataFrame(sqldict)
        
        # Filter the DataFrame to exclude rows where 'Status' contains 'revoke'
        licensed_dataframe = dataframe[~dataframe['Status'].str.contains('Revoked', case=False, na=False)].copy()
        if not licensed_dataframe.empty:
            df2['InternalID_1'] = licensed_dataframe['Approval Number']
            df2['InternalID_1_type'] = 'Approval Number'

            df2['InternalID_2'] = licensed_dataframe['Barcode']
            df2['InternalID_2_type'] = 'Barcode'
            df2['Name'] = licensed_dataframe['Name']
            df2['ListProcessDate'] = processdate
            df2['RegulationDate'] = licensed_dataframe['Approval Date']
            df2['RegCtry'] = reg.split(' ')[0]
            df2['RegCode'] = reg.split(' ')[1]
            df2['ListCode'] = reg.split(' ')[-1]
            df2['ListName'] = Typology[reg]
            df2['RegulationType'] = 'Regulated'  
            df2['Email'] = licensed_dataframe['Email'] 
            df2['Website'] = licensed_dataframe['Website']
        df2 = df2.fillna('')
        
            
        

                

    
    if os.path.exists(tempfolder):

        for rem in os.listdir(tempfolder):

            os.remove(os.path.join(tempfolder, rem))
driver.quit()

df = pd.concat([df1, df2], ignore_index=True)
    

    


Working with list ID BI 1
Working with list ID BI 2


In [7]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

#df=pd.DataFrame(sqldict)

df = df.drop_duplicates()

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_17000\397711335.py:11: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [9]:
df.to_csv('total_list4.csv')

In [11]:
df2

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
4,,,,,,PT Trusted Exchange Nusantara,27/2/KEP.GBI/SR/2025,Approval Number,1567.95-000/Sr,Barcode,...,,,,,,,,,,
5,,,,,,PT Viemo San Thiang Valuta,27/3/KEP.GBI/SR/2025,Approval Number,1569.97-001/Sr,Barcode,...,,,,,,,,,,
6,,,,,,PT Mega Valasindo Perkasa,27/1/KEP.GBI/Btm/2025,Approval Number,1573.214-000/Btm,Barcode,...,,,,,,,,,,
7,,,,,,PT Haloha Valuta Asing,27/1/KEP.GBI/PDG/2025,Approval Number,1377.001/0016,Barcode,...,,,,,,,,,,
8,,,,,,PT Bahtera Perkasa Valasindo,27/1/KEP.GBI/SR/2025,Approval Number,1567.95-000/Sr,Barcode,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4744,,,,,,PT.BANK PEMBANGUNAN DAERAH LAMPUNG,,Approval Number,,Barcode,...,,,,,,,,,,
4745,,,,,,PT.BANK PEMBANGUNAN DAERAH MALUKU DAN MALUKU U...,,Approval Number,,Barcode,...,,,,,,,,,,
4746,,,,,,PT.BANK PEMBANGUNAN DAERAH NTT,,Approval Number,,Barcode,...,,,,,,,,,,
4747,,,,,,PT.BANK PEMBANGUNAN DAERAH PAPUA,,Approval Number,,Barcode,...,,,,,,,,,,


In [20]:
df_combined.to_csv('total_.csv')